In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.ops import nearest_points

# 등고선 데이터 (MultiLineString)
contour_gdf = gpd.read_file("../부산광역시_등고선/부산광역시_등고선_전체.gpkg")

# 군집 포인트 데이터 (Point)
cluster_gdf = gpd.read_file("../../../../../../Downloads/cluster_gdf_all_gu-20250620T125444Z-1-001/cluster_gdf_all_gu/clusters_gdf_all_gu.shp")
# 실제 파일 경로는 환경에 맞게 수정 필요

# 두 GeoDataFrame의 좌표계를 EPSG:4326으로 통일
contour_gdf = contour_gdf.to_crs(epsg=4326)
cluster_gdf = cluster_gdf.to_crs(epsg=4326)

def find_nearest_contour(point, contour_gdf):
    # 주어진 포인트(point)와 등고선 데이터(contour_gdf) 내 모든 라인과의 거리를 계산
    distances = contour_gdf.geometry.distance(point)
    # 가장 가까운 등고선의 인덱스를 찾음
    idx_min = distances.idxmin()
    # 가장 가까운 등고선의 '등고수치' 값을 반환 (등고수치 컬럼명은 실제 데이터에 맞게 수정 필요)
    return contour_gdf.loc[idx_min, '등고수치']

# 각 군집 포인트에 대해 가장 가까운 등고선의 등고수치 값을 찾아 'contour' 컬럼에 저장
cluster_gdf['contour'] = cluster_gdf.geometry.apply(lambda x: find_nearest_contour(x, contour_gdf))

# 결과를 새로운 shapefile로 저장
cluster_gdf.to_file("군집화+등고선.shp")

In [ ]:
# 각 지점별로 등고선 데이터가 추가됐는지 확인
print(cluster_gdf.head())

   cluster_id  data_point                    geometry  contour
0           0        1976   POINT (129.1003 35.21535)     40.0
1           1        1883  POINT (129.15086 35.22258)     55.0
2           2        1435  POINT (129.09771 35.21647)     50.0
3           3        1772  POINT (129.15593 35.23157)     35.0
4           4        2419   POINT (129.1311 35.19808)    125.0
